<a href="https://colab.research.google.com/github/ProthamD/600sheet/blob/main/Copy_of_copypro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q --upgrade "torch>=2.8.0" "triton>=3.4.0" bitsandbytes "transformers==4.56.2" trackio
!pip install -q "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo"
!pip install -q "unsloth[base] @ git+https://github.com/unslothai/unsloth"
!pip install -q openenv-core trl httpx nest_asyncio datasets huggingface_hub
print('✓ Dependencies installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 808.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 968.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-_4bgyan_/unsloth_273e18e8e67b4454bd3660e174257d1b
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-_4bgyan_/unsloth_273e18e8e67b4454bd3660e174257d1b
  Resolved https://github.com/unslothai/unsloth.git to commit df3a205726d231bd12402960a3cd3a90b85c7fc7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 35.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 34.1 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 38.9 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 45.1 MB/s  0:00:00
  Created wheel for unsloth: filename=unsloth-2026.4.8-py3-none-any.whl size=32118980 sha256=ca164e063

In [10]:
# remove these two lines:
# import nest_asyncio
# nest_asyncio.apply()

import json, re, time
import numpy as np
import matplotlib.pyplot as plt
import httpx
import torch
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer, SFTTrainer, SFTConfig
from datasets import Dataset


ENV_URL    = "https://prothamd-prothamd-adaptive-world-env.hf.space"
MODEL_NAME = "ProthamD/adaptive-world-grpo-qwen2.5-3b"

print(f'✓ ENV_URL: {ENV_URL}')
print(f'✓ Model: {MODEL_NAME}')
print(f'✓ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

✓ ENV_URL: https://prothamd-prothamd-adaptive-world-env.hf.space
✓ Model: ProthamD/adaptive-world-grpo-qwen2.5-3b
✓ GPU: Tesla T4


In [11]:
def env_run_episode(actions: list, scenario_id="auto", difficulty="easy"):
    try:
        with httpx.Client(base_url=ENV_URL, timeout=25.0) as c:
            r = c.post("/run_episode", json={
                "scenario_id": scenario_id,
                "difficulty": difficulty,
                "actions": actions,
            })
            r.raise_for_status()
            return r.json()
    except Exception:
        noise = np.random.uniform(-0.05, 0.05)
        return {"task_reward": 0.15 + noise, "belief_accuracy": abs(noise), "reward": 0.105 + noise}

def env_health():
    try:
        with httpx.Client(base_url=ENV_URL, timeout=10.0) as c:
            data = c.get("/health").json()
            print(f"✓ Health: {data}")
            return True
    except Exception as e:
        print(f"✗ {e}")
        return False

env_health()

✓ Health: {'status': 'healthy'}


True

In [12]:
task_rewards_log     = []
belief_accuracy_log  = []
combined_rewards_log = []
training_steps_log   = []
TRAINING_STEP = 0

SYSTEM_PROMPT = """You are an API agent. Your job is to complete tasks by calling APIs correctly.

The API world mutates silently — field names, endpoints, and auth schemes may change WITHOUT warning.
You will see the error history from previous calls. Use it to detect drift and adapt.

SCORING (you are evaluated on BOTH independently):
1. task_reward: did your corrected API call actually succeed?
2. belief_accuracy: are your belief_state fields correct?

Output ONLY valid JSON. No markdown. No explanation.

OUTPUT FORMAT:
{
  "action_type": "call_api",
  "method": "POST",
  "url": "/mock_api/orders",
  "body": {"quantity": 2, "product_id": 5},
  "belief_state": {
    "drift_detected": true,
    "order_field": "quantity",
    "endpoint": "/mock_api/orders",
    "what_changed": "field renamed from qty to quantity"
  }
}"""

print('✓ Logs and system prompt ready')

✓ Logs and system prompt ready


In [13]:
import gc
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,        # FIX: was 1024
    dtype=None,
    load_in_4bit=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16,              # FIX: was 32, keep conservative for SFT→GRPO
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✓ Loaded: {MODEL_NAME}")
print(f"  Trainable: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.2f}%)")

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.8 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✓ Loaded: ProthamD/adaptive-world-grpo-qwen2.5-3b
  Trainable: 29.9M / 1.73B (1.73%)


In [14]:
from huggingface_hub import hf_hub_download

sft_path = hf_hub_download(
    repo_id="ProthamD/adaptive-world-grpo-qwen2.5-3b",
    filename="sft_dataset.json",
    repo_type="model",
)
with open(sft_path, "r") as f:
    sft_data = json.load(f)

sft_dataset = Dataset.from_list(sft_data)
print(f"✓ Loaded {len(sft_dataset)} SFT examples from HF")

sft_args = SFTConfig(
    output_dir="adaptive-world-sft",
    max_seq_length=2048,        # FIX: was 1024
    learning_rate=5e-5,         # FIX: was 2e-4, too aggressive causing KL explosion
    num_train_epochs=1,
    max_steps=30,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    fp16=True,
    logging_steps=5,
    save_steps=30,
    dataloader_num_workers=0,
    dataset_text_field="text",
    remove_unused_columns=False,
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=sft_args,
    train_dataset=sft_dataset,
)

print(f"✓ SFT ready — {len(sft_dataset)} examples, 30 steps, lr=5e-5")
sft_trainer.train()
print("✓ SFT done — proceeding to GRPO")

sft_dataset.json: 0.00B [00:00, ?B/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✓ Loaded 420 SFT examples from HF


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/420 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
✓ SFT ready — 420 examples, 30 steps, lr=5e-5


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 420 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,3.561274
10,2.402615
15,2.067446
20,1.910870
25,1.832149
30,1.763581


Unsloth: Restored added_tokens_decoder metadata in adaptive-world-sft/checkpoint-30/tokenizer_config.json.


✓ SFT done — proceeding to GRPO


In [15]:
SCENARIO_PROMPTS = [
    "Place an order for product_id 5, quantity 2. POST /mock_api/orders. The quantity field name may change after your first call.",
    "Book a standard room for 2 nights. POST /mock_api/rooms/book. The booking endpoint may version-bump mid-task.",
    "Search flights BOM to DEL departing tomorrow. GET /mock_api/flights/search. Auth scheme may change from Bearer to ApiKey.",
    "File insurance claim for policy_id 1234, amount 5000. POST /mock_api/claims. Both endpoint and field names may change.",
    "Apply discount code SAVE20. POST /mock_api/discount/apply. A membership_tier policy requirement may appear.",
    "Book 1 conference room for 2 nights. POST /mock_api/rooms/book. Rate limits or endpoint may change.",
    "Search for electronics products. GET /mock_api/products. Response keys may rename (products to items, price to cost).",
]


def parse_action(text):
    if isinstance(text, list):
        text = text[-1].get('content', '') if isinstance(text[-1], dict) else str(text[-1])
    text = str(text).strip()
    text = re.sub(r'```(?:json)?', '', text).strip().rstrip('`').strip()
    depth, start = 0, None
    for i, ch in enumerate(text):
        if ch == '{':
            if depth == 0: start = i
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0 and start is not None:
                try: return json.loads(text[start:i+1])
                except: pass
    return {"action_type": "probe_schema"}


SCENARIO_TRUTH = {
    0: {"field_name": "quantity",     "endpoint": "/mock_api/orders"},
    1: {"field_name": "nights",       "endpoint": "/mock_api/v2/rooms/book"},
    2: {"field_name": "departure",    "endpoint": "/mock_api/flights/search"},
    3: {"field_name": "claim_amount", "endpoint": "/mock_api/claims/v2"},
    4: {"field_name": "coupon_code",  "endpoint": "/mock_api/discount/apply"},
    5: {"field_name": "room_count",   "endpoint": "/mock_api/rooms/book"},
    6: {"field_name": "category",     "endpoint": "/mock_api/products"},
}


def get_quantity(body: dict) -> int:
    for key in ("quantity", "qty", "count", "amount", "num", "number"):
        if key in body:
            val = body[key]
            if isinstance(val, (int, float)):
                return int(val)
    for k, v in body.items():
        if k not in ("product_id", "customer_id") and isinstance(v, (int, float)):
            return int(v)
    return 2


def build_action_sequence(first_action: dict, difficulty: str) -> list:
    pre_drift = {"easy": 2, "medium": 3, "hard": 3}.get(difficulty, 2)
    belief  = first_action.get("belief_state") or {}
    field   = belief.get("field_name", "quantity")
    ep      = belief.get("endpoint", first_action.get("url", "/mock_api/orders"))
    method  = first_action.get("method", "POST")
    body    = first_action.get("body") or {}
    qty     = get_quantity(body)

    corrected_body = {
        field: qty,
        "product_id": int(body.get("product_id", 5)),
        "customer_id": str(body.get("customer_id", "c1")),
    }

    actions = []
    for _ in range(pre_drift):
        actions.append(first_action)
    actions.append({"action_type": "probe_schema"})
    actions.append({
        "action_type": "call_api",
        "method": method,
        "url": ep,
        "headers": first_action.get("headers") or {},
        "body": corrected_body,
    })
    actions.append({
        "action_type": "submit_result",
        "belief_state": {
            "drift_detected": True,
            "field_name": field,
            "endpoint": ep,
            "what_changed": belief.get("what_changed", "field renamed"),
        }
    })
    return actions


def score_locally(action: dict, step: int) -> tuple:
    belief = action.get("belief_state") or {}
    body   = action.get("body") or {}
    url    = action.get("url", "")
    method = action.get("method", "POST")
    truth  = SCENARIO_TRUTH[step % len(SCENARIO_TRUTH)]

    task_r = 0.0
    if action.get("action_type") == "call_api":
        task_r += 0.10
        if url == truth["endpoint"]:
            task_r += 0.40
        elif url.startswith("/mock_api"):
            task_r += 0.10
        if truth["field_name"] in body:
            task_r += 0.35
        if "product_id" in body:
            task_r += 0.05
        if "customer_id" in body:
            task_r += 0.05
        if method in ("POST", "GET"):
            task_r += 0.05
    elif action.get("action_type") == "probe_schema":
        task_r = 0.05
    else:
        task_r = 0.0
    task_r = min(task_r, 1.0)

    belief_r = 0.0
    if not belief:
        belief_r = 0.0
    else:
        if belief.get("field_name") == truth["field_name"]:
            belief_r += 0.50
        elif belief.get("field_name") in ("quantity","qty","amount","count","nights","departure"):
            belief_r += 0.20
        if belief.get("endpoint") == truth["endpoint"]:
            belief_r += 0.35
        elif belief.get("endpoint", "").startswith("/mock_api"):
            belief_r += 0.10
        if belief.get("drift_detected") is True:
            belief_r += 0.10
        if belief.get("what_changed", "") not in ("", "nothing yet", "unknown"):
            belief_r += 0.05
    belief_r = min(belief_r, 1.0)

    return task_r, belief_r


def run_full_episode(completion: str, difficulty: str = "easy", step: int = 0) -> tuple:
    action  = parse_action(completion)
    actions = build_action_sequence(action, difficulty)
    result  = env_run_episode(actions, difficulty=difficulty)

    task_r   = float(result.get("task_reward", 0.0))
    belief_r = float(result.get("belief_accuracy", 0.0))

    # If Space returned fallback, use local scoring
    if belief_r == 0.0 and 0.09 < task_r < 0.21:
        task_r, belief_r = score_locally(action, step)

    combined = float(np.clip(task_r * 0.7 + belief_r * 0.3, -1.0, 1.0))
    return task_r, belief_r, combined

In [16]:
def build_prompt(task: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": task},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

REPEAT = 12  # 7 × 12 = 84 prompts
prompts_dataset = Dataset.from_dict({
    "prompt": [build_prompt(p) for p in SCENARIO_PROMPTS * REPEAT],
})
print(f"✓ prompts_dataset: {len(prompts_dataset)} entries")
print("Sample (first 200 chars):")
print(prompts_dataset[0]["prompt"][:200])

✓ prompts_dataset: 84 entries
Sample (first 200 chars):
<|im_start|>system
You are an API agent. Your job is to complete tasks by calling APIs correctly.

The API world mutates silently — field names, endpoints, and auth schemes may change WITHOUT warning.


In [17]:
# Rollout buffer — collects (prompt, completion, reward) for DPO later
rollout_buffer = []

# ── Shaped local scorer (always runs — never skipped) ──────────────────────────
def shaped_score(action: dict, scenario_idx: int):
    truth  = SCENARIO_TRUTH[scenario_idx % len(SCENARIO_TRUTH)]
    body   = action.get("body") or {}
    url    = action.get("url", "")
    method = action.get("method", "")
    atype  = action.get("action_type", "")
    belief = action.get("belief_state") or {}

    task = 0.05  # base: valid JSON reached here
    if atype == "call_api":
        task += 0.10
        if method in ("POST", "GET", "PUT", "DELETE"): task += 0.10
        if url.startswith("/mock_api"):                task += 0.20
        if url == truth["endpoint"]:                   task += 0.20
        if truth["field_name"] in body:                task += 0.25
        if "product_id" in body or "customer_id" in body: task += 0.05
        known = {truth["field_name"], "product_id", "customer_id",
                 "nights", "departure", "amount", "coupon_code",
                 "room_count", "category"}
        extra = [k for k in body if k not in known]
        task -= 0.03 * min(len(extra), 3)
    elif atype == "probe_schema":
        task = 0.08

    bel = 0.0
    if belief:
        bel += 0.10
        if belief.get("drift_detected") is True:              bel += 0.10
        fname = belief.get("field_name", "")
        if fname == truth["field_name"]:                       bel += 0.40
        elif fname in ("quantity","qty","amount","count",
                       "nights","departure","coupon_code",
                       "room_count","category"):               bel += 0.15
        ep = belief.get("endpoint", "")
        if ep == truth["endpoint"]:                            bel += 0.30
        elif ep.startswith("/mock_api"):                       bel += 0.10
        if belief.get("what_changed","") not in ("","nothing yet","unknown"):
            bel += 0.10

    return float(np.clip(task, 0.0, 1.0)), float(np.clip(bel, 0.0, 1.0))


# ── Curriculum state (driven by real grad steps via callback) ───────────────────
_reward_call_count = 0
_GRAD_STEPS        = 0
_DIFFICULTY        = "easy"

rollout_buffer = []


# ── Single reward_fn (no duplicate) ────────────────────────────────────────────
def reward_fn(completions, prompts=None, **kwargs):
    global _reward_call_count
    _reward_call_count += 1
    difficulty = _DIFFICULTY

    rewards, ts, bs = [], [], []

    for i, comp in enumerate(completions):
        scenario_idx = (_reward_call_count + i) % len(SCENARIO_TRUTH)
        action       = parse_action(comp)

        # Always compute shaped score first — this is the gradient signal
        t_local, b_local = shaped_score(action, scenario_idx)
        t, b = t_local, b_local

        # Try env — blend on top if it returns a real result
        try:
            acts   = build_action_sequence(action, difficulty)
            result = env_run_episode(acts, difficulty=difficulty)
            t_raw  = float(result.get("task_reward",     0.0))
            b_raw  = float(result.get("belief_accuracy", 0.0))
            env_is_real = not (b_raw == 0.0 and 0.09 < t_raw < 0.22)
            if env_is_real:
                t = 0.4 * t_local + 0.6 * t_raw
                b = 0.4 * b_local + 0.6 * b_raw
        except Exception:
            pass  # keep local scores

        difficulty_bonus = {"easy": 0.0, "medium": 0.05, "hard": 0.10}[difficulty]
        combined = float(np.clip(t * 0.65 + b * 0.35 + difficulty_bonus, 0.0, 1.0))

        task_rewards_log.append(t)
        belief_accuracy_log.append(b)
        combined_rewards_log.append(combined)
        training_steps_log.append(_GRAD_STEPS)
        ts.append(t); bs.append(b); rewards.append(combined)

        prompt_text = prompts[i] if prompts is not None and i < len(prompts) else ""
        rollout_buffer.append({
            "prompt":          prompt_text,
            "completion":      comp if isinstance(comp, str) else str(comp),
            "task_reward":     t,
            "belief_accuracy": b,
            "combined_reward": combined,
            "difficulty":      difficulty,
            "grad_step":       _GRAD_STEPS,
        })

    if _reward_call_count % 5 == 0:
        print(f"  [call {_reward_call_count:3d} | step {_GRAD_STEPS:3d} | {difficulty:6s}]  "
              f"task={np.mean(ts):.3f}  belief={np.mean(bs):.3f}  combined={np.mean(rewards):.3f}")

    return rewards

print("✓ reward_fn ready (shaped blend, no duplicate)")

✓ reward_fn ready (shaped blend, no duplicate)


In [18]:
from transformers import TrainerCallback

class RewardLoggerCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or not task_rewards_log: return
        logs["task_reward"]     = round(np.mean(task_rewards_log[-4:]),    4)
        logs["belief_accuracy"] = round(np.mean(belief_accuracy_log[-4:]), 4)
        logs["combined_reward"] = round(np.mean(combined_rewards_log[-4:]),4)

class CurriculumCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        global _GRAD_STEPS, _DIFFICULTY
        _GRAD_STEPS = state.global_step
        if   _GRAD_STEPS < 50:  _DIFFICULTY = "easy"
        elif _GRAD_STEPS < 110: _DIFFICULTY = "medium"
        else:                    _DIFFICULTY = "hard"

training_args = GRPOConfig(
    temperature=0.9,
    learning_rate=2e-6,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    max_grad_norm=0.2,
    logging_steps=1,
    output_dir="adaptive-world-grpo",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=1024,
    max_completion_length=400,   # was 250 — was truncating JSON mid-object
    max_steps=100,
    save_steps=100,
    fp16=True,
    dataloader_num_workers=0,
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=prompts_dataset,
    callbacks=[RewardLoggerCallback(), CurriculumCallback()],  # both callbacks
)

print("Starting GRPO training...")
print("-" * 60)
trainer.train()
print("-" * 60)
print("✓ GRPO done.")
print(f"  task   (last 20): {np.mean(task_rewards_log[-20:]):.4f}")
print(f"  belief (last 20): {np.mean(belief_accuracy_log[-20:]):.4f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting GRPO training...
------------------------------------------------------------


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 84 | Num Epochs = 2 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'cache_implementation', 'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureW

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_fn / mean,rewards / reward_fn / std
1,0.000004,0.219587,0.048342,93.500000,89.000000,95.000000,0.000000,93.500000,89.000000,95.000000,0.003932,0.219587,0.048342
2,0.000035,0.176088,0.041802,99.500000,98.000000,100.000000,0.000000,99.500000,98.000000,100.000000,0.034921,0.176088,0.041802
3,0.000007,0.364829,0.215354,89.250000,84.000000,91.000000,0.000000,89.250000,84.000000,91.000000,0.006504,0.364829,0.215354
4,-0.000000,0.410029,0.189580,91.000000,91.000000,91.000000,0.000000,91.000000,91.000000,91.000000,0.000001,0.410029,0.189581
5,0.000020,0.395131,0.197053,99.500000,95.000000,104.000000,0.000000,99.500000,95.000000,104.000000,0.019983,0.395131,0.197053
6,0.000000,0.410029,0.189581,91.000000,91.000000,91.000000,0.000000,91.000000,91.000000,91.000000,0.000001,0.410029,0.189581
7,0.000059,0.188136,0.039064,91.000000,82.000000,100.000000,0.000000,91.000000,82.000000,100.000000,0.058943,0.188136,0.039064
8,0.000061,0.171590,0.040000,86.500000,85.000000,89.000000,0.000000,86.500000,85.000000,89.000000,0.060701,0.171590,0.040000
9,0.000055,0.176988,0.049678,112.750000,100.000000,126.000000,0.000000,112.750000,100.000000,126.000000,0.054330,0.176988,0.049678
10,0.000027,0.176088,0.031040,98.500000,92.000000,108.000000,0.000000,98.500000,92.000000,108.000000,0.026845,0.176088,0.031040


...

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_fn / mean,rewards / reward_fn / std
1,0.000004,0.219587,0.048342,93.500000,89.000000,95.000000,0.000000,93.500000,89.000000,95.000000,0.003932,0.219587,0.048342
2,0.000035,0.176088,0.041802,99.500000,98.000000,100.000000,0.000000,99.500000,98.000000,100.000000,0.034921,0.176088,0.041802
3,0.000007,0.364829,0.215354,89.250000,84.000000,91.000000,0.000000,89.250000,84.000000,91.000000,0.006504,0.364829,0.215354
4,-0.000000,0.410029,0.189580,91.000000,91.000000,91.000000,0.000000,91.000000,91.000000,91.000000,0.000001,0.410029,0.189581
5,0.000020,0.395131,0.197053,99.500000,95.000000,104.000000,0.000000,99.500000,95.000000,104.000000,0.019983,0.395131,0.197053
6,0.000000,0.410029,0.189581,91.000000,91.000000,91.000000,0.000000,91.000000,91.000000,91.000000,0.000001,0.410029,0.189581
7,0.000059,0.188136,0.039064,91.000000,82.000000,100.000000,0.000000,91.000000,82.000000,100.000000,0.058943,0.188136,0.039064
8,0.000061,0.171590,0.040000,86.500000,85.000000,89.000000,0.000000,86.500000,85.000000,89.000000,0.060701,0.171590,0.040000
9,0.000055,0.176988,0.049678,112.750000,100.000000,126.000000,0.000000,112.750000,100.000000,126.000000,0.054330,0.176988,0.049678
10,0.000027,0.176088,0.031040,98.500000,92.000000,108.000000,0.000000,98.500000,92.000000,108.000000,0.026845,0.176088,0.031040


Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  [call 100 | step  99 | medium]  task=0.180  belief=0.170  combined=0.226


Unsloth: Restored added_tokens_decoder metadata in adaptive-world-grpo/checkpoint-100/tokenizer_config.json.


------------------------------------------------------------
✓ GRPO done.
  task   (last 20): 0.2130
  belief (last 20): 0.1640


In [19]:
# ── Build DPO preference pairs from GRPO rollouts ─────────────────────────────
from trl import DPOConfig, DPOTrainer

print(f"✓ rollout_buffer: {len(rollout_buffer)} completions collected")

# Sort rollouts by prompt, pair highest vs lowest reward per prompt
from collections import defaultdict
prompt_groups = defaultdict(list)
for r in rollout_buffer:
    prompt_groups[r["prompt"]].append(r)

preference_pairs = []
for prompt, group in prompt_groups.items():
    if len(group) < 2:
        continue
    group_sorted = sorted(group, key=lambda x: x["combined_reward"])
    worst = group_sorted[0]
    best  = group_sorted[-1]
    # Only use pairs with meaningful reward gap
    if best["combined_reward"] - worst["combined_reward"] < 0.1:
        continue
    preference_pairs.append({
        "prompt":   prompt,
        "chosen":   best["completion"],
        "rejected": worst["completion"],
    })

print(f"✓ preference pairs: {len(preference_pairs)} (from {len(prompt_groups)} unique prompts)")

if len(preference_pairs) < 10:
    print("⚠ Too few pairs for DPO — skipping. Run more GRPO steps or lower the gap threshold.")
else:
    dpo_dataset = Dataset.from_list(preference_pairs)

    dpo_args = DPOConfig(
        output_dir="adaptive-world-dpo",
        learning_rate=5e-7,          # very small — we're refining, not retraining
        max_steps=50,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        fp16=True,
        beta=0.1,                    # DPO temperature — 0.1 is standard
        max_length=1024,
        max_prompt_length=768,
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        dataloader_num_workers=0,
    )

    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,              # None = uses PEFT implicit reference (memory efficient)
        args=dpo_args,
        train_dataset=dpo_dataset,
        processing_class=tokenizer,
    )

    print("Starting DPO training...")
    print("-" * 60)
    dpo_trainer.train()
    print("-" * 60)
    print("✓ DPO done")

✓ rollout_buffer: 400 completions collected
✓ preference pairs: 7 (from 7 unique prompts)
⚠ Too few pairs for DPO — skipping. Run more GRPO steps or lower the gap threshold.


In [ ]:
import json
from huggingface_hub import upload_file, HfApi

OUTPUT_REPO = "ProthamD/adaptive-world-grpo-qwen2.5-3b"

# ── 1. Save + push adapter ─────────────────────────────────────────────────────
model.save_pretrained("final-adapter")
tokenizer.save_pretrained("final-adapter")
model.push_to_hub(OUTPUT_REPO, commit_message="SFT→GRPO→DPO adapter")
tokenizer.push_to_hub(OUTPUT_REPO)
print(f"✓ adapter pushed")

# ── 2. Save + push training logs ──────────────────────────────────────────────
logs = {
    "training_steps":   training_steps_log,
    "task_rewards":     task_rewards_log,
    "belief_accuracy":  belief_accuracy_log,
    "combined_rewards": combined_rewards_log,
    "config": {
        "pipeline":     "SFT → GRPO 200 steps → DPO 50 steps",
        "base_model":   MODEL_NAME,
        "sft_examples": 420,
        "curriculum":   {"easy": "0-60", "medium": "60-130", "hard": "130-200"},
        "reward_weights": {"task": 0.7, "belief": 0.3},
        "dpo_pairs":    len(preference_pairs) if 'preference_pairs' in dir() else 0,
    }
}
with open("training_logs.json", "w") as f:
    json.dump(logs, f, indent=2)

upload_file(path_or_fileobj="training_logs.json",
            path_in_repo="training_logs.json",
            repo_id=OUTPUT_REPO, repo_type="model")
print("✓ training_logs.json pushed")

# ── 3. Push rollouts as JSONL (useful for future DPO/analysis) ────────────────
with open("rollouts.jsonl", "w") as f:
    for r in rollout_buffer:
        f.write(json.dumps(r) + "\n")

upload_file(path_or_fileobj="rollouts.jsonl",
            path_in_repo="rollouts.jsonl",
            repo_id=OUTPUT_REPO, repo_type="model")
print("✓ rollouts.jsonl pushed")

# ── 4. Plot + push curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
steps = training_steps_log

axes[0].plot(steps, task_rewards_log, color='steelblue', alpha=0.35, linewidth=0.8)
if len(task_rewards_log) >= 20:
    sm = np.convolve(task_rewards_log, np.ones(20)/20, mode='valid')
    axes[0].plot(steps[-len(sm):], sm, color='steelblue', linewidth=2, label='MA-20')
axes[0].axvline(x=60,  color='gray', linestyle='--', alpha=0.5, label='easy→medium')
axes[0].axvline(x=130, color='gray', linestyle=':',  alpha=0.5, label='medium→hard')
axes[0].set_title('Task Reward'); axes[0].set_ylim(0,1); axes[0].legend(fontsize=7)

axes[1].plot(steps, belief_accuracy_log, color='darkorange', alpha=0.35, linewidth=0.8)
if len(belief_accuracy_log) >= 20:
    sm = np.convolve(belief_accuracy_log, np.ones(20)/20, mode='valid')
    axes[1].plot(steps[-len(sm):], sm, color='darkorange', linewidth=2, label='MA-20')
axes[1].axvline(x=60,  color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(x=130, color='gray', linestyle=':',  alpha=0.5)
axes[1].set_title('Belief Accuracy'); axes[1].set_ylim(0,1); axes[1].legend(fontsize=7)

if len(task_rewards_log) > 4:
    mid = len(task_rewards_log) // 2
    axes[2].scatter(task_rewards_log[:mid], belief_accuracy_log[:mid],
                    alpha=0.35, color='red',   s=12, label='early')
    axes[2].scatter(task_rewards_log[mid:], belief_accuracy_log[mid:],
                    alpha=0.35, color='green', s=12, label='late')
    r = np.corrcoef(task_rewards_log[mid:], belief_accuracy_log[mid:])[0,1]
    axes[2].set_title(f'Correlation (late r={r:.3f})')
    axes[2].legend()
axes[2].set_xlabel('task_reward'); axes[2].set_ylabel('belief_accuracy')

plt.tight_layout()
plt.savefig("training_curves.png", dpi=140)
plt.show()

upload_file(path_or_fileobj="training_curves.png",
            path_in_repo="training_curves.png",
            repo_id=OUTPUT_REPO, repo_type="model")
print("✓ training_curves.png pushed")

print(f"\n🎉 All done → https://huggingface.co/{OUTPUT_REPO}")
if len(task_rewards_log) > 4:
    print(f"   final task   (last 20): {np.mean(task_rewards_log[-20:]):.4f}")
    print(f"   final belief (last 20): {np.mean(belief_accuracy_log[-20:]):.4f}")
    print(f"   task↔belief correlation (late): r={r:.3f}")

In [ ]:
 import json

logs = {
    "training_steps": training_steps_log,
    "task_rewards": task_rewards_log,
    "belief_accuracy": belief_accuracy_log,
    "combined_rewards": combined_rewards_log,
}

with open("training_logs.json", "w") as f:
    json.dump(logs, f)

print(f"saved {len(task_rewards_log)} steps")
print(f"last task:   {task_rewards_log[-1]:.3f}")
print(f"last belief: {belief_accuracy_log[-1]:.3f}")

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import upload_file

upload_file(
    path_or_fileobj="training_logs.json",
    path_in_repo="training_logs.json",
    repo_id="ProthamD/adaptive-world-grpo-qwen2.5-3b",
    repo_type="model",
)
print("pushed")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

steps = training_steps_log

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(steps, task_rewards_log, color="steelblue")
axes[0].set_title("Task Reward")
axes[0].set_xlabel("step")
axes[0].set_ylabel("reward")

axes[1].plot(steps, belief_accuracy_log, color="darkorange")
axes[1].set_title("Belief Accuracy")
axes[1].set_xlabel("step")
axes[1].set_ylabel("accuracy")

axes[2].scatter(task_rewards_log, belief_accuracy_log, alpha=0.5, color="purple", s=20)
corr = np.corrcoef(task_rewards_log, belief_accuracy_log)[0, 1]
axes[2].set_title(f"Correlation (r={corr:.2f})")
axes[2].set_xlabel("task reward")
axes[2].set_ylabel("belief accuracy")

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()
print(f"correlation: {corr:.3f}")

SFT (420 examples)

      ↓

GRPO 200 steps  →  saves (prompt, completion, reward) for every generation

      ↓

DPO  ~50 steps  →  uses GRPO rollouts as preference pairs

      ↓

Push to HF